# Berichte: Word, Excel, PDF

Läuft **offline** mit den mitgelieferten python-docx, XlsxWriter, pypdf, Jinja2 und tabulate (in Jupyter: `pip install python-docx xlsxwriter pypdf jinja2 tabulate`). Die Dateien erscheinen im Dateibereich – öffnen Sie sie mit ↗ in der Textverarbeitung, der Tabellenkalkulation oder dem PDF-Betrachter.

In [ ]:
import pandas as pd
from tabulate import tabulate

sales = pd.DataFrame({'monat': ['Jan', 'Feb', 'Mar'], 'umsatz': [1200, 1350, 1100]})
print(tabulate(sales, headers='keys', tablefmt='github', showindex=False))

In [ ]:
import docx
from jinja2 import Template

text = Template('Umsatz in {{ months }} Monaten: {{ total }} EUR.').render(months=len(sales), total=sales.iloc[:, 1].sum())
doc = docx.Document()
doc.add_heading('Quartalsbericht', 0)
doc.add_paragraph(text)
table = doc.add_table(rows=1, cols=2)
table.rows[0].cells[0].text, table.rows[0].cells[1].text = sales.columns
for m, r in sales.itertuples(index=False):
    row = table.add_row().cells
    row[0].text, row[1].text = m, str(r)
doc.save('bericht.docx')
text

In [ ]:
with pd.ExcelWriter('bericht.xlsx', engine='xlsxwriter') as writer:
    sales.to_excel(writer, sheet_name='Umsatz', index=False)
    sheet = writer.sheets['Umsatz']
    chart = writer.book.add_chart({'type': 'column'})
    chart.add_series({'categories': ['Umsatz', 1, 0, 3, 0], 'values': ['Umsatz', 1, 1, 3, 1]})
    sheet.insert_chart('D2', chart)
print('geschrieben')

In [ ]:
import io
import matplotlib.pyplot as plt
from pypdf import PdfReader, PdfWriter

# zwei einseitige PDFs (aus matplotlib), zu einem zusammengefügt
writer = PdfWriter()
for kind in ('bar', 'line'):
    buf = io.BytesIO()
    sales.plot(x=sales.columns[0], kind=kind, legend=False).figure.savefig(buf, format='pdf')
    plt.close('all')
    writer.append(PdfReader(buf))
writer.write('diagramme.pdf')
len(PdfReader('diagramme.pdf').pages)